In [ ]:
import datetime
import json
import os
import re
import shutil
import zipfile
from pathlib import Path
from typing import List

import nltk
import pandas as pd
import snowflake.snowpark as sp
from snowflake.core import CreateMode, Root
from snowflake.core.table import (
    ForeignKey,
    PrimaryKey,
    Table,
    TableCollection,
    TableColumn,
    TableResource,
)
from snowflake.snowpark import Session, Window
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.context import get_active_session
from trulens.providers.cortex import Cortex

In [ ]:
NLTK_DATA_PATH = Path("nltk_data")
TOKENIZER_PATH = NLTK_DATA_PATH / "tokenizers"
os.environ["NLTK_DATA"] = str(NLTK_DATA_PATH)
nltk.data.path.append(os.environ["NLTK_DATA"])

shutil.rmtree(NLTK_DATA_PATH, ignore_errors=True)
TOKENIZER_PATH.mkdir(parents=True, exist_ok=True)

for file in ("punkt.zip", "punkt_tab.zip"):
    with zipfile.ZipFile(file, mode="r") as zf:
        zf.extractall(TOKENIZER_PATH)

In [ ]:
DEFAULT_DATABASE_NAME = "KNOWLEDGE_BUILDER"
DEFAULT_SCHEMA_NAME = "PUBLIC"
DEFAULT_CSS_NAME = "KB_SEARCH"
DEFAULT_SEARCH_ARGS = dict(columns=["CHUNK_TEXT"], filter={}, limit=5)

In [ ]:
session = get_active_session()
root = Root(session)
database = root.databases[DEFAULT_DATABASE_NAME]
schema = database.schemas[DEFAULT_SCHEMA_NAME]
tc = schema.tables
session.use_schema(f"{DEFAULT_DATABASE_NAME}.{DEFAULT_SCHEMA_NAME}")
CURRENT_USER = session.get_current_user().strip('"')

In [ ]:
def create_table(
    table_collection: TableCollection,
    name: str,
    columns: List[TableColumn],
    mode: CreateMode = CreateMode.error_if_exists,
) -> TableResource:
    table_definition = Table(name=name, columns=columns)
    return table_collection.create(table_definition, mode=mode)


def ingest_golden_pairs(
    session: sp.Session, fp: str, table_name: str, mode: str
) -> sp.DataFrame:
    # In this environment, we don't have a CSV. Mocked from LLM.
    # df = pd.read_csv(fp)
    data = [
        {
            "INCIDENT": "INC0010234",
            "INPUT_QUERY": "I cannot connect to the VPN while working from home.",
            "SUGGESTED_RESOLUTION_CURATED": "Ensure Cisco AnyConnect is updated. Reset your AD password and try reconnecting via the 'Full Tunnel' gateway."
        },
        {
            "INCIDENT": "INC0010567",
            "INPUT_QUERY": "Outlook keeps crashing when I try to open calendar invites.",
            "SUGGESTED_RESOLUTION_CURATED": "Run Outlook in Safe Mode, disable the 'Teams Meeting' add-in, and clear the local AppData cache for Microsoft Office."
        },
        {
            "INCIDENT": "INC0010890",
            "INPUT_QUERY": "Requesting admin access for software installation on Macbook.",
            "SUGGESTED_RESOLUTION_CURATED": "Check if the software is in the Self-Service portal. If not, manager approval is required before granting temporary SUDO rights."
        },
        {
            "INCIDENT": "INC0011221",
            "INPUT_QUERY": "Printer in the London office (2nd floor) is jammed and showing error 404.",
            "SUGGESTED_RESOLUTION_CURATED": "On-site facilities notified. Hardware reset performed; confirmed paper tray 2 was overloaded with A3 paper."
        },
        {
            "INCIDENT": "INC0011456",
            "INPUT_QUERY": "New hire needs access to the Snowflake production warehouse.",
            "SUGGESTED_RESOLUTION_CURATED": "Assign the user to the 'SNOWFLAKE_PROD_READ' Okta group. Sync may take up to 30 minutes to reflect in the Cortex search role."
        }
    ]
    df = pd.DataFrame(data)
    session.write_pandas(df.reset_index(drop=True), 
        table_name, 
        auto_create_table=True, 
        overwrite=True if mode == "overwrite" else False
    )
    return session.table(table_name)


def ingest_synthetic_pairs(
    session: sp.Session, fp: str, table_name: str, mode: str
) -> sp.DataFrame:
    # In this environment, we don't have a CSV. Mocked from LLM.
    # df = pd.read_csv(fp).rename(columns=str.upper) 
    data = [
        {
            "INCIDENT": "INC0012101",
            "INPUT_QUERY": "I received a suspicious email asking for my login credentials.",
            "SUGGESTED_RESOLUTION_CURATED": "Email identified as phishing. Purged from inbox and user directed to complete security awareness training."
        },
        {
            "INCIDENT": "INC0012345",
            "INPUT_QUERY": "My company iPhone is stuck on the Apple logo after an update.",
            "SUGGESTED_RESOLUTION_CURATED": "Perform a forced restart. If unsuccessful, use Apple Configurator to restore the device to factory settings."
        },
        {
            "INCIDENT": "INC0012678",
            "INPUT_QUERY": "Locked out of Workday after three failed login attempts.",
            "SUGGESTED_RESOLUTION_CURATED": "Unlocked account in Active Directory. User advised to clear browser cookies before attempting to log in again."
        },
        {
            "INCIDENT": "INC0012899",
            "INPUT_QUERY": "External monitor is not detecting my laptop via the USB-C dock.",
            "SUGGESTED_RESOLUTION_CURATED": "Update DisplayLink drivers and power cycle the docking station by unplugging it for 30 seconds."
        },
        {
            "INCIDENT": "INC0013112",
            "INPUT_QUERY": "Adobe Creative Cloud says my trial has expired, but I have a license.",
            "SUGGESTED_RESOLUTION_CURATED": "Signed out and back into the Adobe Desktop app using the 'Company Account' SSO option."
        }
    ]
    df = pd.DataFrame(data)
    session.write_pandas(df.reset_index(drop=True), 
        table_name, 
        auto_create_table=True, 
        overwrite=True
    )
    return session.table(table_name)


def search_synthetic_pairs(session: sp.Session, df: sp.DataFrame, current_user: str):
    df = df.select(
        F.parse_json("INPUT_ARGS")["query"].cast(T.StringType()).alias("INPUT_QUERY")
    ).to_pandas()
    for i, row in df.iterrows():
        input_query = row["INPUT_QUERY"]
        search_args = DEFAULT_SEARCH_ARGS.copy()
        search_args.update({"query": input_query})
        resp = css.search(**search_args)
        results = resp.results
        save_search(
            session,
            "SYNTHETIC_PAIR",
            search_args,
            results,
            current_user
        )


def deduplicate_search_results(session: sp.Session) -> None:
    return session.sql("""DELETE FROM SEARCH_QUERIES
    USING
    (
        SELECT SEARCH_ID,
               ROW_NUMBER() OVER 
                    (
                        PARTITION BY INPUT_TYPE, INPUT_ARGS, RESPONSE, CREATED_BY
                        ORDER BY CREATED_ON
                    ) AS ROW_NUM
        FROM SEARCH_QUERIES
        QUALIFY ROW_NUM > 1
    ) AS SRC
    WHERE SEARCH_QUERIES.SEARCH_ID = SRC.SEARCH_ID""").show()


def save_search(
    session: sp.Session, 
    search_type: str, 
    search_args: dict,
    results: dict,
    current_user: str,
) -> None:
    data = [
        [
            search_type,
            search_args,
            results,
            current_user,
            datetime.datetime.now(),
        ]
    ]
    return session.create_dataframe(
        data,
        schema=[
            "INPUT_TYPE",
            "INPUT_ARGS",
            "RESPONSE",
            "CREATED_BY",
            "CREATED_ON",
        ],
    ).write.save_as_table(
        search_queries.fully_qualified_name, mode="append", column_order="name"
    )


def search_golden_pairs(session: sp.Session, df: pd.DataFrame | sp.DataFrame, current_user: str):
    if isinstance(df, sp.DataFrame):
        df = df.to_pandas()
    for i, row in df.iterrows():
        input_query = row["INPUT_QUERY"]
        search_args = DEFAULT_SEARCH_ARGS.copy()
        search_args.update({"query": input_query})
        resp = css.search(**search_args)
        results = resp.results
        save_search(
            session,
            "GOLDEN_PAIR",
            search_args,
            results,
            current_user
        )


def save_feedback(
    session: sp.Session, 
    search_id: int, 
    user_feedback: str, 
    user_rating: int, 
    current_user: str
) -> None:
    current_timestamp = datetime.datetime.now()
    return session.create_dataframe(
        [
            [
                search_id, 
                user_feedback, 
                user_rating, 
                current_user, 
                datetime.datetime.now()
            ]
        ],
        schema=T.StructType(
            [
                T.StructField("SEARCH_ID", T.LongType()),
                T.StructField("USER_FEEDBACK", T.StringType()),  # Changed to String
                T.StructField("USER_RATING", T.LongType()),
                T.StructField("CREATED_BY", T.StringType()),    # Changed to String
                T.StructField("CREATED_ON", T.TimestampType()), # Changed to Timestamp
            ]
        )
    ).write.save_as_table(
        "SEARCH_FEEDBACK", mode="append", column_order="name"
    )


def evaluate(session: sp.Session, model_engine: str, query: str, context: str, types: tuple) -> dict:
    provider = Cortex(session, model_engine=model_engine)
    results = {}
    if "relevance" in types:
        score, reasons = provider.relevance_with_cot_reasons(
            prompt=query,
            response=context
        )
        results["relevance"] = dict(score=score, reasons=reasons)
    if "context_relevance" in types:
        score, reasons = provider.context_relevance_with_cot_reasons(
            question=query,
            context=context
        )
        results["context_relevance"] = dict(score=score, reasons=reasons)
    if "groundedness" in types:
        score, reasons = provider.groundedness_measure_with_cot_reasons(
            source=context,
            statement=context
        )
        results["groundedness"] = dict(score=score, reasons=reasons)
    return results

In [ ]:
search_queries = create_table(
    tc,
    "SEARCH_QUERIES",
    [
        TableColumn(
            name="SEARCH_ID",
            datatype="INT",
            autoincrement=True,
            constraints=[PrimaryKey()],
        ),
        TableColumn(name="INPUT_TYPE", datatype="VARCHAR"),
        TableColumn(name="INPUT_ARGS", datatype="VARIANT"),
        TableColumn(name="RESPONSE", datatype="VARIANT"),
        TableColumn(name="CREATED_BY", datatype="VARCHAR"),
        TableColumn(name="CREATED_ON", datatype="TIMESTAMP_NTZ(9)"),
    ],
    CreateMode.if_not_exists,
)

search_feedback = create_table(
    tc,
    "SEARCH_FEEDBACK",
    [
        TableColumn(
            name="FEEDBACK_ID",
            datatype="INT",
            autoincrement=True,
            constraints=[PrimaryKey()],
        ),
        TableColumn(
            name="SEARCH_ID",
            datatype="INT",
            constraints=[
                ForeignKey(
                    referenced_table_name="SEARCH_QUERIES",
                    referenced_column_names=["SEARCH_ID"],
                )
            ],
        ),
        TableColumn(name="USER_FEEDBACK", datatype="VARCHAR"),
        TableColumn(name="USER_RATING", datatype="INT"),
        TableColumn(name="CREATED_BY", datatype="VARCHAR"),
        TableColumn(name="CREATED_ON", datatype="TIMESTAMP_NTZ(9)"),
    ],
    CreateMode.if_not_exists,
)

golden_pairs = create_table(
    tc,
    "GOLDEN_PAIRS",
    [
        TableColumn(name="INCIDENT", datatype="VARCHAR"),
        TableColumn(name="INPUT_QUERY", datatype="VARCHAR"),
        TableColumn(name="SUGGESTED_RESOLUTION_CURATED", datatype="VARCHAR"),
    ],
    CreateMode.if_not_exists,
)

synthetic_pairs = create_table(
    tc,
    "SYNTHETIC_PAIRS",
    [
        TableColumn(name="SEARCH_ID", datatype="INT"),
        TableColumn(name="INPUT_TYPE", datatype="VARCHAR"),
        TableColumn(name="INPUT_ARGS", datatype="VARCHAR"),
        TableColumn(name="RESPONSE", datatype="VARCHAR"),
        TableColumn(name="CREATED_BY", datatype="VARCHAR"),
        TableColumn(name="CREATED_ON", datatype="TIMESTAMP_NTZ(9)"),
    ],
    CreateMode.if_not_exists,
)

In [ ]:
css = schema.cortex_search_services[DEFAULT_CSS_NAME]
search_args = DEFAULT_SEARCH_ARGS.copy()
search_args.update({"query": "I need to update my computer, right now."})
resp = css.search(**search_args)
results = resp.results

In [ ]:
save_search(
    session,
    "ADHOC",
    search_args,
    results,
    CURRENT_USER,
)

In [ ]:
save_feedback(
    session,
    1,
    "This is a terrible response.",
    1,
    CURRENT_USER,
)

In [ ]:
golden_pairs = ingest_golden_pairs(session, "Golden_Labeling_Pairs(Sheet1) (1).csv", "GOLDEN_PAIRS", "overwrite")

In [ ]:
search_golden_pairs(session, golden_pairs, CURRENT_USER)

In [ ]:
input_query_expr = F.col("INPUT_ARGS")["query"].cast(T.StringType()).alias("INPUT_QUERY")
chunk_text_expr = F.col("VALUE")["CHUNK_TEXT"].cast(T.StringType()).alias("CHUNK_TEXT")
scores_expr = F.col("VALUE")["@scores"].cast(T.VariantType())
cosine_expr = scores_expr["cosine_similarity"].cast(T.FloatType()).alias("COSINE_SIMILARITY")
text_match_expr = scores_expr["text_match"].cast(T.FloatType()).alias("TEXT_MATCH")

flattened_search_responses = (
    session
    .table(search_queries.fully_qualified_name)
    .join_table_function("flatten", F.col("RESPONSE"))
    .select(
        "SEARCH_ID",
        "INPUT_TYPE",
        input_query_expr, 
        chunk_text_expr,
        cosine_expr,
        text_match_expr,
        "CREATED_BY",
        "CREATED_ON",
    )
)
flattened_search_responses.create_or_replace_view("SEARCH_QUERIES_FLATTENED")

In [ ]:
synthetic_pairs = ingest_synthetic_pairs(session, "Untitled 1_2025-12-23-1307.csv", "SYNTHETIC_PAIRS", "overwrite")

In [ ]:
search_golden_pairs(session, synthetic_pairs, CURRENT_USER)

In [ ]:
row = session.table(search_queries.fully_qualified_name).to_pandas().iloc[0]
search_id = row['SEARCH_ID']
input_args = json.loads(row['INPUT_ARGS']) if isinstance(row['INPUT_ARGS'], str) else row['INPUT_ARGS']
response_data = json.loads(row['RESPONSE']) if isinstance(row['RESPONSE'], str) else row['RESPONSE']
query = input_args.get('query', '')
chunks = [r.get('CHUNK_TEXT', '') for r in response_data if isinstance(r, dict)]
context = '\n\n'.join(chunks)
evaluate(session, "llama3.1-70b", query, context, ("relevance", "context_relevance", "groundedness"))